# 1.한글 감성 분석

이 노트북은 **한국어 영화 리뷰 감정분석(NSMC)** 에 대해 **BERT** 를 파인튜닝하는 실습입니다.

## - 학습 진행
- **NSMC 데이터** 다운로드
- `AutoTokenizer`, `AutoModelForSequenceClassification`으로 **토크나이즈 & 모델 구성**
- `Trainer`로 **학습/평가**, 정확도·F1 측정, **혼동행렬** 및 **에러 분석**
- 학습된 모델로 **추론(문장 → 긍/부정)**

## - 사전 준비
- Transformer 계열 모델 사용으로 Colab에서 실습 진행
- 런타임: **GPU 권장** (Colab: 런타임 → 런타임 유형 변경 → 하드웨어 가속기 GPU)
- Python 패키지: `transformers`, `datasets`, `evaluate`, `scikit-learn`, `accelerate`

# 2.감성 분석이란?
- 정의: 텍스트가 나타내는 태도/감정(긍정·부정·중립 등)을 자동으로 분류/점수화하는 작업.
- 적용 예시
    - 리뷰 분석: 쇼핑몰/앱스토어/네이버 영화 리뷰 평판 분석
    - VOC 요약: 고객센터 상담/CS 티켓의 불만 감지·우선순위화
    - 브랜드 모니터링: SNS 여론 추이, 캠페인 효과 측정
    - 콘텐츠 필터링: 부적절/혐오/악성 댓글 탐지

# 3.한국어 감성 분석의 특징
- 교착어/띄어쓰기 변동: “좋다/좋아요/좋았음” 등 다양한 활용형, 띄어쓰기 오류 빈번
- 대화체/은어/줄임말: “노잼”, “개좋”, “현웃”
- 이모지/반어/비꼼: “역시 기대 이하네요^^”, “그냥 최고… (반어 가능)”
- 도메인 차이: 영화/게임/푸드/패션마다 어휘가 다름 → 도메인 적합화 필요

# 4.전체 파이프라인
1. 문제 정의 (이진/다중/점수? 라벨 기준?)
2. 데이터 수집/라벨링 (NSMC, 사내 로그, 크라우드 라벨링)
3. 데이터 전처리 (결측/중복/이상치, 정규식, 이모지 처리)
4. 표현(Feature Engineering/Embedding)
    - 전통: BoW/TF-IDF, n-gram
    - 분산표현: Word2Vec/FastText
    - 사전학습모델: BERT/RoBERTa/ELECTRA 계열(한국어 특화: KoELECTRA, KLUE-RoBERTa, KcBERT, KoBERT)
5. 모델링
    - 전통 ML: 로지스틱회귀, SVM
    - 딥러닝: 트랜스포머 파인튜닝
6. 학습/튜닝 (train/valid/test, 하이퍼파라미터 조정)
7. 평가 (정확도, F1, PR-AUC, 혼동행렬 등)
8. 배포/모니터링

# 5.실습

## 0)런타임 준비
- Colab의 런타임 유형을 GPU로 바꾸면 모델 학습 단계가 더 빨라집니다.
- 베이스라인은 CPU로도 충분합니다.

In [1]:
import torch, platform, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA available: True
Device: Tesla T4


- 구글 마운트

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1)환경 세팅
- 전통 ML: scikit-learn
- 딥러닝: transformers, datasets, evaluate, accelerate

In [3]:
%pip -q install scikit-learn tqdm emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.7 MB/s eta 0:00:00


In [4]:
%pip -q install -U "transformers>=4.44,<4.47" "datasets>=2.20" "accelerate>=0.34" "evaluate>=0.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 57.3 MB/s eta 0:00:00


## 2)데이터 다운로드
- 네이버 영화평 데이터 : https://github.com/e9t/nsmc
- GitHub에서 받아 압축 해제 후 판다스로 읽습니다.

In [5]:
import os, pandas as pd

# 데이터 다운로드
!rm -rf nsmc && git clone -q https://github.com/e9t/nsmc.git
data_dir = "/content/nsmc"

In [28]:
train_path = os.path.join(data_dir, '/content/nsmc/ratings_train.txt')
test_path  = os.path.join(data_dir, '/content/nsmc/ratings_test.txt')

train_df = pd.read_csv(train_path, sep='\t')
test_df  = pd.read_csv(test_path,  sep='\t')

In [29]:
train_df.head(), train_df.shape, test_df.shape

(         id                                           document  label
 0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
 1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
 2  10265843                                  너무재밓었다그래서보는것을추천한다      0
 3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
 4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1,
 (150000, 3),
 (50000, 3))

## 3)데이터 전처리
- 한글/숫자/기본 문장부호만 남깁니다. 공란/결측 제거도 합니다.
- 정규표현식을 이용해 불필요 문자 제거 → 공백 정리.

In [30]:
import re
import numpy as np

def clean_korean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # 한글, 숫자, 기본 문장부호만 남기기
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.\,\!\?\-~…]", " ", text)
    # 연속 공백 정리
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 결측/공백 제거
train_df = train_df.dropna(subset=['document']).copy()
test_df  = test_df.dropna(subset=['document']).copy()
train_df['document'] = train_df['document'].map(clean_korean)
test_df['document']  = test_df['document'].map(clean_korean)

# 빈 문자열 제거
train_df = train_df[train_df['document'].str.len() > 0]
test_df  = test_df[test_df['document'].str.len() > 0]

In [31]:
train_df.shape, test_df.shape

((149619, 3), (49849, 3))

In [10]:
train_df.tail()

,id,document,label
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1
149999,9619869,한국 영화 최초로 수간하는 내용이 담긴 영화,0


In [11]:
test_df.tail()

,id,document,label
49995,4608761,오랜만에 평점 로긴했네 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함,1
49996,5308387,의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO,0
49997,9072549,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49998,5802125,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0
49999,6070594,마무리는 또 왜이래,0


## 4)데이터 분리(검증셋 만들기)
- 과대적합을 막고, 하이퍼파라미터를 조정할 기준이 필요합니다.
- 따라서, 훈련 데이터를 train/valid로 분리합니다.

In [32]:
from sklearn.model_selection import train_test_split

train_texts = train_df['document'].tolist()
train_labels = train_df['label'].tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    train_texts,
    train_labels,
    test_size=0.1,          # train:val = 9:1
    random_state=1004,
    stratify=train_labels
)

len(X_train), len(X_valid)

(134657, 14962)

## 5)베이스라인: TF-IDF + 로지스틱 회귀
- 빠르게 “기준 성능”을 얻기 위해. 학습/추론이 매우 빠르고 해석도 쉽습니다.

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

In [34]:
tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=3,               # 전체 문서에서 등장 문서 수가 3 미만인 단어는 제거
        max_df=0.95,            # 너무 보편적인 단어(상위 5%)는 제거(정보량 낮음)
        ngram_range=(1,2),      # uni + bi-gram
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=200,           # 최적화 반복 횟수
        C=4.0,                  # C=1이 디폴트, 규제 강도(작을 수록 규제 강함). 4.0은 비교적 규제가 약한 편.
        n_jobs=None if hasattr(LogisticRegression(), "n_jobs") else None
    ))
])

- 모델 학습

In [35]:
tfidf_lr.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, min_df=3, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf', LogisticRegression(C=4.0, max_iter=200))])

- 검증 데이터셋 평가

In [36]:
# 검증셋 평가
valid_pred = tfidf_lr.predict(X_valid)
print("Validation accuracy:", accuracy_score(y_valid, valid_pred))
print(classification_report(y_valid, valid_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_valid, valid_pred))

Validation accuracy: 0.8116561956957626
              precision    recall  f1-score   support

           0     0.7882    0.8534    0.8195      7498
           1     0.8394    0.7697    0.8030      7464

    accuracy                         0.8117     14962
   macro avg     0.8138    0.8116    0.8113     14962
weighted avg     0.8138    0.8117    0.8113     14962

Confusion Matrix:
 [[6399 1099]
 [1719 5745]]


- 테스트셋 최종 평가

In [37]:
test_pred = tfidf_lr.predict(test_df['document'])
print("Test accuracy:", accuracy_score(test_df['label'], test_pred))
print(classification_report(test_df['label'], test_pred, digits=4))

Test accuracy: 0.8138578507091416
              precision    recall  f1-score   support

           0     0.7904    0.8507    0.8194     24748
           1     0.8408    0.7776    0.8079     25101

    accuracy                         0.8139     49849
   macro avg     0.8156    0.8141    0.8137     49849
weighted avg     0.8158    0.8139    0.8136     49849



- 간단 추론 함수

In [38]:
def predict_sentiment(text: str):
    cleaned = clean_korean(text)
    proba = tfidf_lr.predict_proba([cleaned])[0]
    label = int(proba[1] >= 0.5)
    return {"text": text, "pred": label, "neg_prob": float(proba[0]), "pos_prob": float(proba[1])}


In [19]:
examples = [
    "노잼 핵노잼 도리도리노잼스. 이 영화 나가라 안 본다.",
    "장난하나 이런 걸 영화라고 만들어놨노. 돌았나.",
    "",
    "와",
    "뭐고"
]
for ex in examples:
    print(predict_sentiment(ex))

{'text': '노잼 핵노잼 도리도리노잼스. 이 영화 나가라 안 본다.', 'pred': 0, 'neg_prob': 0.9946309066026662, 'pos_prob': 0.005369093397333838}
{'text': '장난하나 이런 걸 영화라고 만들어놨노. 돌았나.', 'pred': 0, 'neg_prob': 0.9193577078558179, 'pos_prob': 0.08064229214418211}
{'text': '', 'pred': 0, 'neg_prob': 0.5232874913198678, 'pos_prob': 0.47671250868013215}
{'text': '와', 'pred': 0, 'neg_prob': 0.5232874913198678, 'pos_prob': 0.47671250868013215}
{'text': '뭐고', 'pred': 0, 'neg_prob': 0.687078523999084, 'pos_prob': 0.3129214760009161}


## 6)트랜스포머 파인튜닝(Fine-tuning)
- 한국어에 특화된 사전학습 모델을 파인튜닝하면 베이스라인 대비 성능이 더 잘 나옵니다.


In [39]:
import os, pandas as pd, numpy as np, torch, evaluate
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split

In [40]:
# --- 빠른 실험용 샘플링: 전체의 40%만 사용 ---
frac = 0.35
train_df, _ = train_test_split(
    train_df,
    train_size=frac,
    stratify=train_df["label"],
    random_state=1004
)
print(train_df.shape, test_df.shape)

(52366, 3) (49849, 3)


- 토크나이저/모델
    - https://huggingface.co/beomi/KcELECTRA-base

In [41]:
# 토크나이저/모델: beomi/KcELECTRA-base
model_name = 'beomi/KcELECTRA-base'                     # 한국어 댓글/커뮤니티 데이터에 강한 ELECTRA 계열 모델

tokenizer = AutoTokenizer.from_pretrained(model_name)   # 모델 종류에 맞는 토크나이저 클래스를 자동으로 선택

def tokenize_fn(batch):
    return tokenizer(batch["document"], padding="max_length", truncation=True, max_length=96)

dataset = DatasetDict(
    train=Dataset.from_pandas(train_df[["document","label"]]),
    test =Dataset.from_pandas(test_df[["document","label"]])
)
tokenized = dataset.map(tokenize_fn, batched=True, num_proc=2, remove_columns=["document"])

Map (num_proc=2):   0%|          | 0/52366 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/49849 [00:00<?, ? examples/s]

In [24]:
# num_labels=2 : 감성 분류 라벨이 2개. 긍/부정
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

accuracy = evaluate.load("accuracy"); f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred          # logits : 모델 출력값, labels : 실제 정답
    preds = np.argmax(logits, axis=1)
    return {"acc": accuracy.compute(predictions=preds, references=labels)["accuracy"],
            "f1":  f1.compute(predictions=preds, references=labels, average="macro")["f1"]}

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [42]:
# --- 빠른 학습 설정 ---
training_args = TrainingArguments(
    output_dir="./out_fast",
    num_train_epochs=1,
    per_device_train_batch_size=64,    # T4면 32~64 사이에서 맞추기, GPU 1개당 학습 배치 크기
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    eval_strategy="no",         # 학습 중간 평가 하지 X, 마지막에 따로 평가
    save_strategy="no",         # 학습 중 체크포인트 저장 X,
    logging_steps=200,          # 200 step마다 학습 로그를 출력
    report_to="none",           # wandb 같은 외부 로깅 도구 사용 X
)

# Trainer : 모델 학습, 평가, 예측 과정은 편하게 실행주는 Hugging Face 클래스
trainer = Trainer(
    model=model,                                # 학습할 사전학습(PLM) 모델
    args=training_args,                         # 위에서 정의한 학습 파라미터
    train_dataset=tokenized["train"],           # 학습 데이터 셋
    eval_dataset=tokenized["test"],             # 최종만 보고 싶으면 evaluate()만 호출
    tokenizer=tokenizer,                        # 토크나이저
    compute_metrics=compute_metrics             # 평가 지표 함수(F1, ACC 등)
)

/tmp/ipykernel_7253/3330334233.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


- 모델 학습

In [43]:
trainer.train()
metrics = trainer.evaluate(tokenized["test"])
metrics

Step,Training Loss
200,0.261900
400,0.244600
600,0.234500
800,0.239600


{'eval_loss': 0.2512136399745941,
 'eval_acc': 0.9040502316997332,
 'eval_f1': 0.9040337135543176,
 'eval_runtime': 266.2405,
 'eval_samples_per_second': 187.233,
 'eval_steps_per_second': 2.926,
 'epoch': 1.0}

- 추론 함수

In [44]:
id2label = {0: "NEG", 1: "POS"}

def infer_transformer(texts):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    if torch.cuda.is_available():
        model.to("cuda")
        enc = {k: v.to("cuda") for k, v in enc.items()}
    with torch.no_grad():
        model_inputs = {k: v for k, v in enc.items() if k != 'token_type_ids' or 'token_type_ids' in model.forward.__code__.co_varnames}
        out = model(**model_inputs).logits
        prob = out.softmax(dim=-1).cpu().numpy()
    preds = prob.argmax(axis=1)
    return [{"text": t, "pred": int(p), "label": id2label[int(p)], "neg_prob": float(pr[0]), "pos_prob": float(pr[1])}
            for t, p, pr in zip(texts, preds, prob)]

In [45]:
infer_transformer([
    "돈주고 봐야되는 영화가 아니라 돈받고 봐야되는 희대의 괴작"
])

[{'text': '돈주고 봐야되는 영화가 아니라 돈받고 봐야되는 희대의 괴작',
  'pred': 0,
  'label': 'NEG',
  'neg_prob': 0.9751970767974854,
  'pos_prob': 0.02480296976864338}]

## 07)모델 저장하기

In [46]:
import joblib
import torch, torch.nn.functional as F

In [47]:
# 경로 설정
# 구글 마운트 했다면 내 드라이브에 저장 가능
os.makedirs("artifacts", exist_ok=True)
os.makedirs("model", exist_ok=True)

# 전통 ML 모델 저장
joblib.dump(tfidf_lr, "artifacts/tfidf_lr_nsms.joblib")
print("TF-IDF + Logistic Regression 모델 저장 완료")

TF-IDF + Logistic Regression 모델 저장 완료


In [49]:
# 트랜스포머 모델 및 토크나이저 저장
save_dir = "drive/MyDrive/saved_model_nsmc"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"트랜스포머 모델과 토크나이저가 저장되었습니다: {save_dir}")

트랜스포머 모델과 토크나이저가 저장되었습니다: drive/MyDrive/saved_model_nsmc


- 추론

In [50]:
load_dir = "./saved_model_nsmc"
tokenizer = AutoTokenizer.from_pretrained(load_dir)
model = AutoModelForSequenceClassification.from_pretrained(load_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

def predict_sentiment(texts):
    # texts: str 또는 list[str]
    if isinstance(texts, str):
        texts = [texts]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=96
    ).to(device)

    model_inputs = {k: v for k, v in inputs.items() if k != 'token_type_ids' or 'token_type_ids' in model.forward.__code__.co_varnames}

    with torch.no_grad():
        logits = model(**model_inputs).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy() # Move to CPU for numpy conversion
        preds = probs.argmax(axis=1)

    results = []
    for text, pred, prob in zip(texts, preds, probs):
        results.append({
            "text": text,
            "pred": int(pred), # Store as integer
            "label": "긍정" if int(pred) == 1 else "부정",
            "pos_prob": float(prob[1]),
            "neg_prob": float(prob[0])
        })
    return results

In [51]:
examples = [
    "저는 앞이 보이지 않지만 이 영화를 보았습니다.",
    "이동준씨가 60억 가까이 날리고 밤무대 똥꼬(?) 차력쇼까지 해야했던...눈물겨운 영화...시갈 형님은 마지막 5분가량 출연 10억 챙기고 주연까지..영화보면 알게된다..평점으로 패러독스와 반전의 미학을 이끌어낼 수있다는 경의로움을...",
    "남자는 세번 운다고한다....태어날때.부모님 돌아가실때.나라 망할때...허나 나는 클레멘 타인은 보고 생각이 바꿨다... 남자는 세번 운다....클레멘타인은 볼때....클레멘타인 결말은 볼때...클레멘타인은 본것 후회할때...",
    "로또1등이 되면 제일 먼저 하고 싶은일은? 이란 질문에 저도 모르게 피식 웃었습니다. 클레멘타인은 로또1등이 안되어도 볼수 있었기 때문이죠"
]

for r in predict_sentiment(examples):
    print(f"[{r['label']}] {r['text']}")
    print(f"  🔹 긍정확률: {r['pos_prob']:.3f}, 부정확률: {r['neg_prob']:.3f}\n")

[긍정] 저는 앞이 보이지 않지만 이 영화를 보았습니다.
  🔹 긍정확률: 0.991, 부정확률: 0.009

[긍정] 이동준씨가 60억 가까이 날리고 밤무대 똥꼬(?) 차력쇼까지 해야했던...눈물겨운 영화...시갈 형님은 마지막 5분가량 출연 10억 챙기고 주연까지..영화보면 알게된다..평점으로 패러독스와 반전의 미학을 이끌어낼 수있다는 경의로움을...
  🔹 긍정확률: 0.675, 부정확률: 0.325

[긍정] 남자는 세번 운다고한다....태어날때.부모님 돌아가실때.나라 망할때...허나 나는 클레멘 타인은 보고 생각이 바꿨다... 남자는 세번 운다....클레멘타인은 볼때....클레멘타인 결말은 볼때...클레멘타인은 본것 후회할때...
  🔹 긍정확률: 0.734, 부정확률: 0.266

[부정] 로또1등이 되면 제일 먼저 하고 싶은일은? 이란 질문에 저도 모르게 피식 웃었습니다. 클레멘타인은 로또1등이 안되어도 볼수 있었기 때문이죠
  🔹 긍정확률: 0.150, 부정확률: 0.850

